Check out the Training notebook in my [Training Notebook](https://www.kaggle.com/code/mehulkumar99/spider-question-to-sql-query)

Check out the Agent notebook in my [Agent Notebook](https://www.kaggle.com/code/mehulkumar99/agentic-loop-addressing-syntax-errors)

Check out the RAG notebook in my [RAG](https://www.kaggle.com/code/mehulkumar99/rag-sql)

## Ablation table (Accuracy Progression)


| Step | Change | Execution Accuracy |
|---|---|---|
| Phi-2 2.7B baseline | Unaugmented schema | 40% |
| Llama 3.1 8B | Unaugmented schema | 70.89% |
| + Column order fix | Normalise result tuple order | 72% |
| + Augmented inference | Sample rows, 142 examples | 75% |
| + Token count fix | All 301 wrong/error examples | 77.27% |
| + Augmented fine-tuning | 6,607 training examples | 81% |
| + Agentic loop + CoT | Chain-of-thought + retry on all failures | 84.14% |
| + Evaluation bug fix | Column name normalisation | **85.59%** |
| + LangGraph refactor | StateGraph with conditional retry edges | 85.59% (structural improvement) |



# NL2SQL Inference + Evaluation Pipeline

## Overview
Runs the fine-tuned model on the Spider validation set and measures execution accuracy against ground truth databases.

---

## Data Flow
The inference and evaluation pipeline processes data through the following stages:

1. **Validation Data (`validation_df`)** Compiles 1,034 validation examples.
2. **Prompt Construction (`build_prompt()`)** Injects the augmented schema into the prompt with `inference=True` (no response target).
3. **Batching (`DataLoader`)** Batches inputs with a `batch_size=8` using left-padding for generation.
4. **Generation (`model.generate()`)** Generates SQL queries using greedy decoding with `max_new_tokens=128`.
5. **SQL Execution (`execute_sql()`)** Runs the predicted SQL query against the target database.
6. **Normalization (`normalize_df()`)** standardizes the output dataframes for fair comparison. Ensures `Colummn Order Invariant` result comparison
7. **Comparison (`execution_match()`)** Checks the normalized predicted result against the ground truth result.
8. **Parallel Evaluation (`execution_accuracy_parallel()`)** Distributes the workload across a `ThreadPoolExecutor` with 4 workers.
9. **Final Metric** Achieves an **81% execution accuracy**.

---

## Key Design Decisions
* **Throughput & Determinism:** Utilizes batched inference for speed paired with greedy decoding to ensure deterministic, reproducible outputs.
* **Order-Invariant Result Comparison:** Normalizes dataframes before evaluation by sorting columns alphabetically and rows by value to prevent false negatives caused by ordering discrepancies.
* **Parallel Evaluation:** Speeds up evaluation by testing independent examples in parallel via multi-threading

In [1]:
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

!pip install --upgrade huggingface_hub -q
!pip install "torchvision>=0.27.0" --upgrade -q
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" --force-reinstall --no-cache-dir -q
!pip install --no-deps "trl<0.13.0" peft accelerate -q

!pip install -q "numpy==2.0.2"
!pip install -q "Pillow==11.3.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 188.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 264.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 299.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 183.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 283.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 264.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 330.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 210.7 MB/s eta 0:00:00a 0

In [2]:
!pip install bitsandbytes --upgrade -q

In [3]:
import ctypes
ctypes.CDLL('/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13')
import bitsandbytes

In [ ]:
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 -q
# !pip install --upgrade huggingface_hub -q
# !pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" -q
# !pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" --force-reinstall --no-cache-dir -q
# !pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate -q
# !pip install "bitsandbytes==0.46.0" --force-reinstall --no-cache-dir -q
# !pip install -q "numpy==2.0.2"
# !pip install -q "Pillow==11.3.0"

In [4]:
import warnings
import transformers
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

In [5]:
! pip show unsloth

Name: unsloth
Version: 2026.5.8
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: https://unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: nest-asyncio, pydantic, pyyaml, typer
Required-by: 


# Loading Model 

In [6]:
# # CELL 2 - restart kernel after cell 1, then run this
# import os
# if not os.path.exists("/kaggle/working/llama-3-1-clean"):
#     from huggingface_hub import snapshot_download
#     snapshot_download(
#         repo_id="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
#         local_dir="/kaggle/working/llama-3-1-clean",
#         local_dir_use_symlinks=False,
#         resume_download=True
#     )

In [14]:
import os
# Redirect the temporary download cache to the larger temp folder
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"

from huggingface_hub import snapshot_download

# This will pick up exactly where it left off (at 3.1GB) and finish it safely
snapshot_download(
    repo_id="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    local_dir="/kaggle/working/llama-3-1-clean",
    local_dir_use_symlinks=False,
    resume_download=True
)

'/kaggle/working/llama-3-1-clean'

In [13]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    # Points to the official, verified 4-bit Llama 3.1 Instruct model hosted by Unsloth
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,          # Automatically detects your Kaggle GPU type (T4 or P100)
    load_in_4bit=True,
    # device_map={"": 0} # Unsloth handles device placement automatically; you can omit this
)

# Enable Unsloth's optimized native inference mode (Makes it 2x faster)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Exception: EOF while parsing a list at line 176703 column 1

In [11]:
# from unsloth import FastLanguageModel
# import torch

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="/kaggle/working/llama-3-1-clean",
#     max_seq_length=2048,
#     dtype=None,
#     load_in_4bit=True,
#     device_map={"": 0}
# )

RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

In [ ]:
# Apply LoRA adapters
from peft import PeftModel
adapter_path = '/kaggle/input/notebooks/mehulkumar99/spider-question-to-sql-query/outputs/checkpoint-450'
model = PeftModel.from_pretrained(model, adapter_path)

print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB")
model.print_trainable_parameters()

# Loading validation set

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

dataset = load_dataset("xlangai/spider")
# validation_df = pd.DataFrame(dataset['validation'])
train_df = pd.DataFrame(dataset['train'])


validation_df = pd.read_csv('/kaggle/input/datasets/mehulkumar99/error-sql-prediction-csv/validation_predictions.csv')
validation_df = validation_df.drop(columns=['query_toks', 'query_toks_no_value','question_toks',])
validation_df.head()
# validation_df.iloc[361]

In [ ]:
wrong_data = validation_df.loc[validation_df['results'].isin(['error', 'wrong'])]
wrong_error_index = wrong_data.index.tolist()

In [ ]:
len(wrong_error_index)

# Loading schema dataset

In [ ]:
spider_tables = load_dataset("richardr1126/spider-schema", split="train")
df= pd.DataFrame(spider_tables)

print(spider_tables)

In [1]:
import pickle

# The path will now be in /kaggle/input/ instead of /kaggle/working/
file_path = '/kaggle/input/datasets/mehulkumar99/augmented-schema/schema_lookup_augmented.pkl'

with open(file_path, 'rb') as f:
    schema_lookup = pickle.load(f)

print(f"Loaded schema for {len(schema_lookup)} databases.")

Loaded schema for 166 databases.


In [2]:
print(schema_lookup['car_1']['augmented_schema'])

# Table: continents 
## Columns:
- ContId (number, PK) | e.g: [1, 2, 3]
- Continent (text) | e.g: ['america', 'europe', 'asia']

# Table: countries 
## Columns:
- CountryId (number, PK) | e.g: [1, 2, 3]
- CountryName (text) | e.g: ['usa', 'germany', 'france']
- Continent (number, FK -> continents.ContId) | e.g: [1, 2, 2]

# Table: car_makers 
## Columns:
- Id (number, PK) | e.g: [1, 2, 3]
- Maker (text) | e.g: ['amc', 'volkswagen', 'bmw']
- FullName (text) | e.g: ['American Motor Company', 'Volkswagen', 'BMW']
- Country (text, FK -> countries.CountryId) | e.g: ['1', '2', '2']

# Table: model_list 
## Columns:
- ModelId (number, PK) | e.g: [1, 2, 3]
- Maker (number, FK -> car_makers.Id) | e.g: [1, 2, 3]
- Model (text) | e.g: ['amc', 'audi', 'bmw']

# Table: car_names 
## Columns:
- MakeId (number, PK) | e.g: [1, 2, 3]
- Model (text, FK -> model_list.Model) | e.g: ['chevrolet', 'buick', 'plymouth']
- Make (text) | e.g: ['chevrolet chevelle malibu', 'buick skylark 320', 'plymouth satellit

In [ ]:
print(schema_lookup.keys())

## Understanding table Counts

In [ ]:
import numpy as np
def total_tables(schema):
    count = 0
    count = sum(1 for char in schema if char =='|')
    return count+1

import matplotlib.pyplot as plt
import numpy as np

# 1. Process the data
df['count_tables'] = df['Schema (values (type))'].apply(total_tables)
train_df['schema'] = train_df['db_id'].apply(lambda x: schema_lookup[x]['Schema (values (type))'])
train_df['count_tables'] = train_df['schema'].apply(total_tables)
validation_df['schema'] = validation_df['db_id'].apply(lambda x: schema_lookup[x]['Schema (values (type))'])
validation_df['count_tables'] = validation_df['schema'].apply(total_tables)

# 2. Define plotting data
datasets = [
    (df['count_tables'], 'Per Database'),
    (train_df['count_tables'], 'Training Set'),
    (validation_df['count_tables'], 'Validation Set')
]

# 3. Create subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (data, title) in zip(axes, datasets):
    ax.hist(data, bins=10, edgecolor='black', color='skyblue')
    ax.set_title(f'Table Count: {title}')
    ax.set_xlabel('Table Count')
    ax.set_ylabel('Frequency')
    print(f'Mean count ({title}): {np.mean(data):.2f}')

plt.tight_layout()
plt.show()

In [ ]:
print(schema_lookup['concert_singer']['augmented_schema'])

# BUILDING PROMPT

In [ ]:
def build_prompt( schema_lookup, index, dataset, inference = False, schema_type = None ):

    df = dataset
    prompt = ''

    db_id = df['db_id'][index]
    # print(db_id)
    question = df['question'][index]

    if not schema_type:
        schema = schema_lookup[db_id]['format_schema']
    else:
        schema = schema_lookup[db_id]['augmented_schema']
    
    prompt = f"""<|start_header_id|>system<|end_header_id|>

Convert the natural language question to SQL using the schema below. Use SQLite syntax and SQLite functions only.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Schema:
{schema}

### Question:
{question}"""

    if inference == True:
        text = prompt +  f"""<|eot_id|>""" + "<|start_header_id|>assistant<|end_header_id|>\n\n"
        return {'text': text}


    elif inference == False:
        query = df['query'][index]
        text = prompt+ f"""<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{query}<|eot_id|>"""
        
        return {'text':text}
    return{'text': prompt}



text = build_prompt( schema_lookup, index =1293,dataset = train_df, inference = True, schema_type = 1 )
print(text['text'])
print(len(text['text']))
    

In [ ]:
# Pick one soccer_1 example
idx = train_df[train_df['db_id'] == 'baseball_1'].index[0]
prompt = build_prompt(schema_lookup, index=idx, dataset=train_df,
                     inference=False, schema_type= 1)['text']

tokens = tokenizer(prompt)['input_ids']
print(f"Char length  : {len(prompt)}")
print(f"Token count  : {len(tokens)}")
print(f"Chars/token  : {len(prompt)/len(tokens):.2f}")

# See how individual parts tokenize
print(f"\nTokens for 'Player_Attributes': {len(tokenizer('Player_Attributes')['input_ids'])}")
print(f"Tokens for '| e.g: [1, 2, 3]' : {len(tokenizer('| e.g: [1, 2, 3]')['input_ids'])}")

## Filtering augmented tokens larger than 1950. 

In [ ]:
train_df['original_index'] = train_df.index

train_df = train_df.reset_index(drop=True)  # align positional and label index

lengths = []
for i in range(len(train_df)):
    # prompt = build_prompt(schema_lookup, index=i, dataset=train_df, 
    #                      inference=False, schema_type=1)['text']
    prompt = build_prompt(schema_lookup, index=i, dataset=train_df, 
                     inference=False, schema_type= 1)['text']
    
    lengths.append(len(tokenizer(prompt)['input_ids']))

train_df['T'] = lengths

filtered_df = train_df[train_df['T'] > 2048]
print(f"Over 2048: {len(filtered_df)}")
print(filtered_df['T'].describe())

In [ ]:
filtered_df.groupby('db_id').agg(
    example_count=('T', 'count'),
    avg_tokens=('T', 'mean'),
    max_tokens=('T', 'max'),
).sort_values('avg_tokens', ascending=False)

In [ ]:
train_df.to_csv('/kaggle/working/train_df_with_tokens.csv', index=False)

In [ ]:
train_df.to_pickle('/kaggle/working/train_df_with_tokens.pkl')

In [ ]:
filtered_df.head(3)

In [ ]:
validation_df = validation_df.reset_index(drop=True)  # align positional and label index

lengths = []
for i in range(len(validation_df)):
    prompt = build_prompt(schema_lookup, index=i, dataset=validation_df, 
                         inference= True, schema_type=1)['text']
    # prompt = build_prompt(schema_lookup, index=i, dataset=train_df, 
    #                  inference=False, schema_type= None)['text']
    
    lengths.append(len(tokenizer(prompt)['input_ids']))

validation_df['T'] = lengths

filtered_df = validation_df[validation_df['T'] > 2048]
print(f"Over 2048: {len(filtered_df)}")
print(filtered_df['T'].describe())

In [ ]:
#schema_lookup, index, dataset, last_exec_sql = None, error_msg = None, inference = False 
total = len(validation_df)
lengths = []
for i in range(total):
    prompt = build_prompt( schema_lookup, index = i , dataset = validation_df, inference = True, schema_type = 1 )['text']
    lengths.append(len(tokenizer(prompt)['input_ids']))

validation_df['T'] = lengths
# & (validation_df['T']>1950)
filtered_df = validation_df[(validation_df['results'].isin(['error', 'wrong'])) ]

# wrong_erro_df = filtered_df.loc[filtered_df['results'].isin(['error', 'wrong'])]
filtered_sorted_df = filtered_df.sort_values(by='T', ascending = False)

# print(len(idx_prompt_limit_exceeded))
filtered_sorted_df.head()

In [ ]:
import re
filtered_sorted_df['column_count'] = filtered_sorted_df['db_id'].apply(
    lambda db_id: sum(
        len(re.findall(r'^- ', block, re.MULTILINE))
        for block in re.split(r'(?=# Table:)', schema_lookup[db_id]['augmented_schema'].strip())
        if block.strip()
    )
)

filtered_sorted_df['table_count'] = filtered_sorted_df['db_id'].apply(
    lambda db_id: len(re.findall(r'^# Table:', schema_lookup[db_id]['augmented_schema'], re.MULTILINE))
)
filtered_sorted_df.to_csv("/kaggle/working/filtered_sorted_df.csv", index=False)

# filtered_sorted_df[['db_id', 'T', 'table_count', 'column_count']].head(20)

In [ ]:
filtered_sorted_df.groupby('db_id').agg(
    example_count=('T', 'count'),
    avg_tokens=('T', 'mean'),
    max_tokens=('T', 'max'),
    table_count=('table_count', 'first'),
    column_count=('column_count', 'first')
).sort_values('avg_tokens', ascending=False).round(0)

In [ ]:
print(len(filtered_sorted_df))
filtered_index = filtered_sorted_df.index.tolist()


## single inference check

In [15]:
import torch
from unsloth import FastLanguageModel

# 1. Load your successfully downloaded local model
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/llama-3-1-clean",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Enable Unsloth's 2x faster inference mode
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [29]:
# Apply LoRA adapters
from peft import PeftModel
adapter_path = '/kaggle/input/notebooks/mehulkumar99/spider-question-to-sql-query/outputs/checkpoint-450'
model = PeftModel.from_pretrained(model, adapter_path)

print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB")
model.print_trainable_parameters()

Memory used: 5.76 GB
trainable params: 0 || all params: 8,072,204,288 || trainable%: 0.0000


In [30]:


import torch

def build_prompt( schema_lookup, question,db_id, inference = False, schema_type = None ):

    
    prompt = ''


    # print(db_id)
 

    if not schema_type:
        schema = schema_lookup[db_id]['format_schema']
    else:
        schema = schema_lookup[db_id]['augmented_schema']
    
    prompt = f"""<|start_header_id|>system<|end_header_id|>

Convert the natural language question to SQL using the schema below. Use SQLite syntax and SQLite functions only.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Schema:
{schema}

### Question:
{question}"""

    if inference == True:
        text = prompt +  f"""<|eot_id|>""" + "<|start_header_id|>assistant<|end_header_id|>\n\n"
        return {'text': text}


    elif inference == False:
        query = df['query'][index]
        text = prompt+ f"""<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{query}<|eot_id|>"""
        
        return {'text':text}
    return{'text': prompt}


# 1. Build the prompt for a single question
question = 'Show the song name and release year of the song by the youngest singer'
db_id = 'concert_singer'

result = build_prompt(schema_lookup,question,db_id ,inference=True, schema_type=1)
single_text = result['text']

# 2. Tokenize the single prompt
tokenizer.padding_side = 'left' 

inputs = tokenizer(
    single_text,
    truncation=True,
    max_length=2048,
    add_special_tokens=True,
    return_tensors='pt'  # Returns PyTorch tensors directly
)

input_ids = inputs['input_ids'].to(model.device)
attention_mask = inputs['attention_mask'].to(model.device)

# 3. Generate the SQL output
model.eval()
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,        # Greedy search
        use_cache=True,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        pad_token_id=tokenizer.pad_token_id,
    )

# 4. Extract and clean the generated SQL
# Slice out the prompt tokens (equivalent to input_ids.shape[1] since batch size is 1)
generated = outputs[0][input_ids.shape[1]:]
sql = tokenizer.decode(generated, skip_special_tokens=True).strip()

# Post-processing to match your loop logic
if sql.startswith("assistant"):
    sql = sql[len("assistant"):].strip()

print("Generated SQL:\n", sql)

Generated SQL:
 SELECT song_name,  song_release_year FROM singer ORDER BY age LIMIT 1


# DATA LOADER PREP

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Step 1 — build prompts WITHOUT the answer (inference=True)
inference_texts = []
for i in filtered_index:
    result = build_prompt(schema_lookup, i, validation_df, inference=True, schema_type =1)
    inference_texts.append(result['text'])

# Step 2 — tokenize all prompts
tokenized = tokenizer(
    inference_texts,
    truncation=True,
    max_length=2048,
    padding=False,          # don't pad yet — DataLoader will handle per batch
    add_special_tokens= True,
    return_tensors=None,    # return list, not tensors
)

# Step 3 — build a simple dataset
from datasets import Dataset
inference_dataset = Dataset.from_dict({
    'input_ids': tokenized['input_ids'],
    'attention_mask': tokenized['attention_mask'],
})

tokenizer.padding_side = 'left'

# Step 4 — use DataCollatorWithPadding to batch and pad
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors='pt',
)

dataloader = DataLoader(
    inference_dataset,
    batch_size=8,
    shuffle=False,           # keep order so predictions align with val_df rows
    collate_fn=collator,
)

In [ ]:
# from torch.utils.data import DataLoader
# from transformers import DataCollatorWithPadding

# # Step 1 — build prompts WITHOUT the answer (inference=True)
# inference_texts = []
# for i in range(len(validation_df)):
#     result = build_prompt(schema_lookup, i, validation_df, inference=True)
#     inference_texts.append(result['text'])

In [ ]:
# tokenizer.padding_side = 'left'
# tokenized = tokenizer(
#     inference_texts,
#     truncation=True,
#     max_length=2048,
#     padding=False,          # don't pad yet — DataLoader will handle per batch
#     add_special_tokens= True,
#     return_tensors=None,    # return list, not tensors
# )
# print(len(tokenized['input_ids']))
# print(list(tokenized.keys()))

Dataset.from_dict takes the keys from your dictionary (input_ids, attention_mask) and turns them into Columns. Each item in the list becomes a Row.

Row 1: input_ids for your first SQL prompt, attention_mask for your first SQL prompt.

Row 2: input_ids for your second SQL prompt

In [ ]:
# from datasets import Dataset
# inference_dataset = Dataset.from_dict({
#     'input_ids': tokenized['input_ids'],
#     'attention_mask': tokenized['attention_mask'],
# })
# print(inference_dataset.column_names)
# print(len(inference_dataset))

In [ ]:
# collator = DataCollatorWithPadding(
#     tokenizer=tokenizer,
#     padding=True,
#     return_tensors='pt',
# )
# print(collator.tokenizer.pad_token_id) # shows the pad_token_id used by the token

In [ ]:
# dataloader = DataLoader(
#     inference_dataset,
#     batch_size=16,
#     shuffle=False,           # keep order so predictions align with val_df rows
#     collate_fn=collator,
# )

In [ ]:
# 1. Grab one batch
batch = next(iter(dataloader))

# 2. Get the pad token ID from your tokenizer
pad_id = tokenizer.pad_token_id

# 3. Inspect the first few tokens of the first row in the batch
# If it's Left Padding, the first few numbers should be the pad_id
print(f"Pad Token ID: {pad_id}")
print(f"First 10 tokens of row 0: {batch['input_ids'][1][:10].tolist()}")
print(f"Last 10 tokens of row 0:  {batch['input_ids'][1][-10:].tolist()}")

# 4. Logical check
is_left = batch['input_ids'][0][0] == pad_id
print(f"\nIs it Left Padded? {is_left}")

print(f'length of the first batch: {len(batch['input_ids'][1])}')

# GENERATION

## single example

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import time
torch.cuda.reset_peak_memory_stats()

start = time.time()
batch = next(iter(dataloader))
input_ids = batch['input_ids'].to(model.device)
attention_mask = batch['attention_mask'].to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        pad_token_id=tokenizer.pad_token_id,
    )

elapsed = time.time() - start
peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"Time: {elapsed:.1f}s, Peak memory: {peak:.2f}GB")

In [ ]:
!pip show bitsandbytes

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# from unsloth import FastLanguageModel

# FastLanguageModel.for_inference(model)  # unsloth inference mode — 2x faster

from tqdm.auto import tqdm
predictions = []

model.eval()
for batch in tqdm(dataloader, desc = 'Generating'):
    input_ids      = batch['input_ids'].to(model.device)
    attention_mask = batch['attention_mask'].to(model.device)

    # GREEDY SEARCH
    with torch.no_grad():
        outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,        # greedy
        use_cache=True,         # works fine with greedy
        eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        pad_token_id=tokenizer.pad_token_id,
        # temperature=0.2, # Very close to greedy but allows slight variation
        # top_p=0.95,
    )

    for i, output in enumerate(outputs):
        # print(output)
        actual_prompt_len = batch['attention_mask'][i].sum().item()
        generated = output[batch['input_ids'].shape[1]:]
        sql = tokenizer.decode(generated, skip_special_tokens=True).strip()
        if sql.startswith("assistant"):
            sql = sql[len("assistant"):].strip()
        predictions.append(sql)


print(predictions[:14])

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# import json
# with open("/kaggle/working/predictions.json", "w") as f:
#     json.dump(predictions, f)

# Save as CSV too (backup)
pd.DataFrame(predictions).to_csv("/kaggle/working/predictions.csv", index=False)

In [ ]:
validation_df['new_pred_sql'] = validation_df['predicted_sql']

# Assuming 'df' is your DataFrame and 'indices' is your list
validation_df.loc[ filtered_index, 'new_pred_sql'] = predictions 


In [ ]:
index = 767
print(validation_df['predicted_sql'].iloc[index])
print(validation_df['new_pred_sql'].iloc[index])

In [ ]:

filtered_index

In [ ]:
validation_df.head(3)

In [ ]:
# # Save predictions
# import pandas as pd
# validation_df['predicted_sql'] = predictions
validation_df.to_csv("/kaggle/working/new_validation_predictions.csv", index=False)

# # Save model adapter too if not already saved
# model.save_pretrained("/kaggle/working/final_adapter")
# tokenizer.save_pretrained("/kaggle/working/final_adapter")

In [ ]:
# schema_table= pd.DataFrame(spider_tables)
# schema_table['format_schema'] = df['db_id'].apply(format_schema)

In [ ]:
import pandas as pd
validation_df = pd.read_csv('/kaggle/working/new_validation_predictions.csv')

validation_df = pd.read_csv('/kaggle/input/notebooks/mehulkumar99/agentic-loop/CoT_agent_validation_predictions.csv')
validation_df.head(3)

In [ ]:
validation_df['results'].value_counts()

# EXECUTING SQL

In [ ]:
import os
import sqlite3

KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
def execute_sql(sql, db_id, db_dir = KAGGLE_DB_DIR ):
    
    """Execute SQL against SQLite DB, return result set or None on error."""
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

    # CRITICAL: Check if the file actually exists before connecting
    if not os.path.exists(db_path):
        print(f"Warning: Database file not found at {db_path}")
        return None
    try:
        # 'mode=ro' tells SQLite to open it in Read-Only mode
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result
    # except Exception as e:
    #     return None   # invalid SQL
    except sqlite3.Error as e:
        # This will return the specific SQLite error message (e.g., "no such column: name")
        print(f"SQL Error on {db_id}: {e}")
        return str(e)
    except Exception as e:
        # This catches other issues (like file path errors)
        print(f"General Error: {e}")
        return None
    
sql = 'SELECT T1.Continent,  count(*),  T2.ContinentId FROM continents AS T1 JOIN countries AS T2 ON T1.ContId  =  T2.Continent GROUP BY T2.ContinentId'
result = execute_sql(sql, db_id = 'car_1')
print(result)

In [ ]:
import os
import sqlite3
import pandas as pd

def normalize_df(df):
    """Sort columns alphabetically and rows by all columns for order-invariant comparison."""
    df = df.copy()
    # Normalize column names to lowercase
    df.columns = [c.lower().strip() for c in df.columns]
    # Sort columns alphabetically
    df = df.reindex(sorted(df.columns), axis=1)
    # Sort rows by all columns (converts to str to handle mixed types)
    df = df.apply(lambda col: col.astype(str).str.strip().str.lower())
    df = df.sort_values(by=list(df.columns)).reset_index(drop=True)
    return df

def execution_match(pred_df, gold_df):
    """Returns True if pred and gold have same content regardless of column/row order."""
    try:
        pred_norm = normalize_df(pred_df)
        gold_norm = normalize_df(gold_df)

        # print(pred_norm)
        # print(gold_norm)

        pred_norm.columns = range(len(pred_norm.columns))
        gold_norm.columns = range(len(gold_norm.columns))
                
        if pred_norm.equals(gold_norm):
            return 'correct'
    except Exception:
        return 'wrong'

def evaluate_pair(db_path, pred_sql, gold_sql):
    conn = sqlite3.connect(db_path)
    try:
        pred_df = pd.read_sql_query(pred_sql, conn)
        gold_df = pd.read_sql_query(gold_sql, conn)
        return execution_match(pred_df, gold_df)
    except Exception as e:
        return 'error'  # SQL error = wrong
    finally:
        conn.close()

# pred_sql = 'SELECT T1.Continent,  count(*),  T1.ContId FROM continents AS T1 JOIN countries AS T2 ON T1.ContId  =  T2.Continent GROUP BY T1.ContId'
# gold_sql = 'SELECT T1.ContId ,  T1.Continent ,  count(*) FROM CONTINENTS AS T1 JOIN COUNTRIES AS T2 ON T1.ContId  =  T2.Continent GROUP BY T1.ContId'

KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
db_dir = KAGGLE_DB_DIR

i = 0

pred_sql = validation_df['new_pred_sql'].iloc[i]
gold_sql = validation_df['query'].iloc[i]
db_id = validation_df['db_id'].iloc[i]

db_path = db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

results = evaluate_pair(db_path, pred_sql, gold_sql)
results

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
import threading
from tqdm.auto import tqdm

def execution_match(pred_df, gold_df):
    pred_norm = normalize_df(pred_df)
    gold_norm = normalize_df(gold_df)
    return 'correct' if pred_norm.equals(gold_norm) else 'wrong'

def evaluate_pair(db_path, pred_sql, gold_sql):
    conn = sqlite3.connect(db_path, check_same_thread=False, timeout=10)
    try:
        pred_df = pd.read_sql_query(pred_sql, conn)
        gold_df = pd.read_sql_query(gold_sql, conn)
        return execution_match(pred_df, gold_df)
    except Exception:
        return 'error'
    finally:
        conn.close()

def evaluate_single(args):
    i, pred_sql, gold_sql, db_id = args
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")
    status = evaluate_pair(db_path, pred_sql, gold_sql)
    return i, status

def execution_accuracy_parallel(validation_df, max_workers=4):
    total = len(validation_df)
    args_list = [
        (i,
         validation_df['new_pred_sql'].iloc[i],
         validation_df['query'].iloc[i],
         validation_df['db_id'].iloc[i])
        for i in range(total)
    ]

    results = [None] * total

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(evaluate_single, args): args[0] for args in args_list}
        for future in tqdm(as_completed(futures), total=total, desc="Evaluating"):
            i = futures[future]
            try:
                i, status = future.result()
                results[i] = status
            except Exception as e:
                results[i] = 'error'

    correct  = sum(1 for r in results if r == 'correct')
    errors   = sum(1 for r in results if r == 'error')
    wrong    = sum(1 for r in results if r == 'wrong')
    accuracy = correct / total

    print(f"Execution Accuracy : {accuracy:.4f} ({correct}/{total})")
    print(f"Wrong              : {wrong}/{total}")
    print(f"Invalid SQL errors : {errors}/{total}")
    return accuracy, results

In [ ]:
# Step 2 — evaluation (CPU, parallelized, no GPU needed)
accuracy, results = execution_accuracy_parallel(validation_df, max_workers=4)


In [ ]:
error = sum(1 for i, result in enumerate(results) if result == 'error')
error

In [ ]:
validation_df['results'] = results


In [ ]:
validation_df.to_csv("/kaggle/working/newest_validation_predictions.csv", index=False)

In [ ]:
validation_df['results'].value_counts()